# Lab 06: Your First RAG Chain -- SOLUTION

**Goal:** Build a complete RAG chain that retrieves relevant documents and generates grounded answers using LCEL.

**What you'll learn:**
- How to combine a retriever + prompt + LLM into a RAG chain
- What RunnablePassthrough does in a RAG chain
- How context injection works in the prompt
- The complete data flow from question to answer

## Step 1: Set up the knowledge base

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
import os
import huggingface_hub

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    huggingface_hub.login(token=hf_token, add_to_git_credential=False)


In [ ]:
documents = [
    Document(page_content="Annual leave is 24 days per year. Unused leave cannot be carried forward. Apply through the internal portal at least 3 days in advance.",
             metadata={"source": "handbook.pdf", "page": 5}),
    Document(page_content="Sick leave is 12 days per year. A medical certificate is required for absences of more than 2 consecutive days.",
             metadata={"source": "handbook.pdf", "page": 5}),
    Document(page_content="Maternity leave is 26 weeks as per government regulations. Paternity leave is 2 weeks. Apply at least 30 days in advance.",
             metadata={"source": "handbook.pdf", "page": 6}),
    Document(page_content="Employees can work from home up to 3 days per week with team lead approval. Core hours are 10 AM to 4 PM IST.",
             metadata={"source": "handbook.pdf", "page": 8}),
    Document(page_content="VPN connection is mandatory for accessing internal systems from home. Contact IT helpdesk for VPN setup assistance.",
             metadata={"source": "handbook.pdf", "page": 8}),
    Document(page_content="Internet reimbursement of Rs 1,500 per month for WFH employees. Submit broadband bill to finance by the 5th of each month.",
             metadata={"source": "handbook.pdf", "page": 9}),
    Document(page_content="Travel expenses must be submitted with original receipts within 7 days. Meal allowance during client visits is Rs 500 per day.",
             metadata={"source": "handbook.pdf", "page": 12}),
    Document(page_content="Laptops are provided by the company and replaced every 3 years. Software license requests go through the IT helpdesk.",
             metadata={"source": "tech-guide.pdf", "page": 7}),
    Document(page_content="Tech stack: Python (FastAPI), Java (Spring Boot) for backend. React, Angular for frontend. PostgreSQL, MongoDB for databases. AWS for cloud.",
             metadata={"source": "tech-guide.pdf", "page": 3}),
    Document(page_content="Bangalore office: WeWork Embassy Tech Village, 5th Floor (HQ, 200+ employees). Mumbai office: Worli Business District, Tower A, 12th Floor.",
             metadata={"source": "handbook.pdf", "page": 15}),
]

print("Building knowledge base...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Knowledge base ready: {vectorstore._collection.count()} documents")

## Step 2: Create the LLM

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile")
print("LLM ready (Groq)")

## Step 3: Create the RAG prompt and build the chain

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful company assistant. Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say "I don't have that information in our handbook."

Context:
{context}

Question: {question}

Answer:"""
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built!")

## Step 4: Ask questions

In [ ]:
questions = [
    "How many days of annual leave do I get?",
    "Can I work from home?",
    "What is the meal allowance for client visits?",
    "Where is the Bangalore office?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")

## Step 5: Test with an out-of-scope question

In [ ]:
q = "What is the company's stock price?"
print(f"Q: {q}")
print(f"A: {rag_chain.invoke(q)}")

## TODO 1: Debug Retrieval Quality (SOLUTION)

In [ ]:
query = "How do I expense a client dinner?"

# Step 1: See what the retriever found
retrieved = retriever.invoke(query)
print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved):
    print(f"  {i+1}. [{doc.metadata.get('source', '?')}, p.{doc.metadata.get('page', '?')}]")
    print(f"     {doc.page_content[:100]}...\n")

# Step 2: Get the RAG chain's answer
answer = rag_chain.invoke(query)
print(f"RAG Answer: {answer}")

# Step 3: Try a query where retrieval might struggle
tricky_query = "What is the dress code?"
tricky_docs = retriever.invoke(tricky_query)
print(f"\n--- Tricky query: '{tricky_query}' ---")
print(f"Retrieved chunks (likely irrelevant):")
for doc in tricky_docs:
    print(f"  - {doc.page_content[:80]}...")
print(f"\nRAG Answer: {rag_chain.invoke(tricky_query)}")
print("(Good RAG chains say 'I don't know' when context is irrelevant!)")

## TODO 2: Add Confidence Scoring to RAG (SOLUTION)

In [ ]:
# Thresholds are Chroma squared-L2 distances (lower = better match), calibrated
# against THIS corpus: real hits land at 0.44-1.06, off-topic questions at 1.42+.
def rag_with_confidence(question, threshold_high=1.1, threshold_low=1.4):
    """Answer a question and report confidence based on retrieval scores."""
    results = vectorstore.similarity_search_with_score(question, k=3)

    if not results:
        return {"answer": "No documents found.", "confidence": "NONE", "best_score": None}

    best_score = results[0][1]
    docs = [doc for doc, score in results]
    context = "\n\n".join(doc.page_content for doc in docs)

    if best_score <= threshold_high:
        confidence = "HIGH"
    elif best_score <= threshold_low:
        confidence = "MEDIUM"
    else:
        confidence = "LOW"

    answer = (rag_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return {"answer": answer, "confidence": confidence, "best_score": round(best_score, 4)}


# Questions spanning all three confidence bands: answered, partially related, out of scope
for q in [
    "How many sick days do I get?",
    "Where is the Mumbai office?",
    "Can I expense my lunch at the office?",
    "What is the company's stock price?",
]:
    result = rag_with_confidence(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   Confidence: {result['confidence']} (score: {result['best_score']})\n")

## Key Takeaways

- **RAG chain** = retriever | format | prompt | llm | parser
- **RunnablePassthrough** passes the question through unchanged
- The LLM only uses the context you provide -- no hallucination
- **Debug retrieval** by inspecting retrieved docs separately
- **Confidence scoring** uses similarity scores to flag unreliable answers